# Qwen3-TTS audio renderer (jelly-jlpt)
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. The first run only renders **one test line per character** so you can hear every voice before committing.

When you are happy: open the last cell, uncomment it (select all, `Ctrl+/`) and run it. It renders every clip (about 370), writes `manifest.json`, zips `audio/` and downloads it.

**Upload `render-input.json`** when the settings cell asks (it is in `.scratch/colab-tts/`).

In [ ]:
!pip install -q -U qwen-tts soundfile

In [ ]:
import torch, gc, json, numpy as np, soundfile as sf
from IPython.display import Audio, display
from google.colab import files as files_mod
from qwen_tts import Qwen3TTSModel
print(torch.cuda.get_device_name(0))

def load(name):  # T4: float16 (no bfloat16), no flash-attn
    return Qwen3TTSModel.from_pretrained(name, device_map="cuda:0", dtype=torch.float16)

_models = {}
def model(kind):  # loaded on first use
    ids = {"custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
           "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
           "base":   "Qwen/Qwen3-TTS-12Hz-1.7B-Base"}
    if kind not in _models: _models[kind] = load(ids[kind])
    return _models[kind]

def preset(speaker, instruct=None): return {"kind": "preset", "speaker": speaker, "instruct": instruct}
def design(description):            return {"kind": "design", "description": description}

def raw(spec, text, seed):
    torch.manual_seed(seed)
    if spec["kind"] == "preset":
        kw = {"instruct": spec["instruct"]} if spec.get("instruct") else {}
        w, sr = model("custom").generate_custom_voice(text=text, language="Japanese", speaker=spec["speaker"], **kw)
    else:
        w, sr = model("design").generate_voice_design(text=text, language="Japanese", instruct=spec["description"])
    return np.asarray(w[0], dtype=np.float32), sr

def locked_speaker(spec, ref_text, seed):
    """speak(text, seed) -> (wave, sr), always through a clone of one reference line, so the voice cannot drift."""
    ref_wave, sr = raw(spec, ref_text, seed)
    prompt = model("base").create_voice_clone_prompt(ref_audio=(ref_wave, sr), ref_text=ref_text)
    def speak(text, s):
        torch.manual_seed(s)
        w, sr2 = model("base").generate_voice_clone(text=text, language="Japanese", voice_clone_prompt=prompt)
        return np.asarray(w[0], dtype=np.float32), sr2
    return speak

def loudness(w, target_db):
    """Scale a line so its speech (silence ignored) sits at target_db RMS; never clip."""
    active = w[np.abs(w) > 0.02 * np.abs(w).max()]
    g = 10 ** (target_db / 20) / (np.sqrt(np.mean(active ** 2)) + 1e-9)
    w = w * g
    peak = np.abs(w).max()
    return w * (0.97 / peak) if peak > 0.97 else w

## Settings (the values you approved) and the input file

In [ ]:
SEED      = 1234
TARGET_DB = -20
GAIN_DB   = {"F": 1, "M": 0, "N": 0}

# Fixed reference sentence every voice is cloned from. Same for all characters.
REF_TEXT = "おはようございます。今日は天気がいいので、駅まで歩いて行きます。"

NARRATOR = preset("Serena", "Speak slowly and clearly in a calm, neutral tone like an exam recording.")

print("Upload render-input.json")
up = files_mod.upload()
DATA = json.loads(next(iter(up.values())).decode("utf-8"))
TRACKS, MAN, WOMAN = DATA["tracks"], DATA["manArchetypes"], DATA["womanArchetypes"]
print(len(TRACKS), "tracks")

## Preview: one line per character
Listen to every voice here before running the full render.

In [ ]:
# characters actually used by the tracks
used = {("N", "")}
for t in TRACKS:
    if t["man"]:   used.add(("M", t["man"]))
    if t["woman"]: used.add(("F", t["woman"]))

def spec_for(role, arch):
    if role == "N": return NARRATOR
    if role == "M": return design(MAN[arch])
    return preset("Ono_Anna", WOMAN[arch])

SPEAKERS = {}
for role, arch in sorted(used):
    SPEAKERS[(role, arch)] = locked_speaker(spec_for(role, arch), REF_TEXT, SEED)

def sample_line(role, arch):
    for t in TRACKS:                      # first real line this character speaks
        if role == "N" or t["man" if role == "M" else "woman"] == arch:
            for l in t["lines"]:
                if l["role"] == role and len(l["text"]) > 6:
                    return l["text"]
    return "今日はいい天気ですね。"

LABEL = {"N": "Narrator", "M": "Man", "F": "Woman"}
for role, arch in sorted(used):
    text = sample_line(role, arch)
    w, sr = SPEAKERS[(role, arch)](text, SEED)
    w = loudness(w, TARGET_DB + GAIN_DB.get(role, 0))
    print(LABEL[role], "|", arch or "fixed", "|", text)
    display(Audio(w, rate=sr))

## Full run
Commented out on purpose. Uncomment (select all, `Ctrl+/`) once the previews sound right.

In [ ]:
# # FULL RUN: renders every unique clip, writes manifest.json, zips and downloads.
# # Uncomment this whole cell (select all, Ctrl+/), then run it. Safe to re-run: finished clips are skipped.
# import os, json, hashlib, zlib, subprocess, time

# OUT = "/content/audio"
# os.makedirs(OUT, exist_ok=True)

# def arch_of(track, role):
#     return {"M": track["man"], "F": track["woman"]}.get(role, "")

# def clip_key(track, role, text):          # same rule as tools/audio/README.md
#     return f"{role}|{arch_of(track, role)}|{text}"

# done, t0 = 0, time.time()
# todo = {}                                 # key -> (role, arch, text)
# for t in TRACKS:
#     for l in t["lines"]:
#         todo.setdefault(clip_key(t, l["role"], l["text"]), (l["role"], arch_of(t, l["role"]), l["text"]))
# print(len(todo), "unique clips to render")

# names = {}
# for key, (role, arch, text) in todo.items():
#     name = hashlib.sha1(key.encode("utf-8")).hexdigest()[:12] + ".mp3"
#     names[key] = name
#     path = f"{OUT}/{name}"
#     if os.path.exists(path):
#         continue
#     w, sr = SPEAKERS[(role, arch)](text, SEED + zlib.crc32(key.encode("utf-8")) % 100000)
#     w = loudness(w, TARGET_DB + GAIN_DB.get(role, 0))
#     sf.write("/content/_tmp.wav", w, sr)
#     subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/content/_tmp.wav", "-ac", "1", "-ar", "24000", "-b:a", "48k", path], check=True)
#     done += 1
#     if done % 20 == 0:
#         print(f"{done} rendered, {int(time.time() - t0)} s")

# manifest = {"version": 1, "tracks": {}}   # per-track file list, same order as the play sequence
# for t in TRACKS:
#     manifest["tracks"][t["id"]] = [names[clip_key(t, l["role"], l["text"])] for l in t["lines"]]
# json.dump(manifest, open(f"{OUT}/manifest.json", "w"), ensure_ascii=False, indent=1)

# total = sum(os.path.getsize(f"{OUT}/{n}") for n in set(names.values()))
# print(f"{len(set(names.values()))} clips, {total / 1e6:.1f} MB")
# !cd /content && rm -f audio.zip && zip -qr audio.zip audio
# files_mod.download("/content/audio.zip")